# Figura 5 · De la nómina bruta a lo que llega a la cuenta

Este cuaderno crea la Figura 5 del trabajo con **Plotly**. El resultado es un bloque de código que se pega dentro de `index.html`, en el hueco de la Figura 5 (la web carga la librería Plotly una sola vez, en su cabecera).

**Qué muestra.** Para cada sueldo bruto anual, de 10.000 a 100.000 €, en qué se reparte cada 100 € de la nómina: lo que se lleva la **Seguridad Social**, lo que se lleva el **IRPF** y lo que **llega a la cuenta** (el neto). Con unos botones se elige el año (2015, 2019 o 2024). Al pasar el ratón se ven los euros de cada parte y el tramo más alto del IRPF que alcanza ese sueldo. Dos líneas marcan el salario mediano y el medio de cada año.

**Datos.** La tabla `data/processed/curva_bruto_neto.csv`, que genera el cuaderno principal con el modelo fiscal del repositorio, y los sueldos de `data/processed/salarios_ccaa.csv`. No se crea ninguna tabla nueva.

**Supuestos** (los del modelo fiscal): asalariado soltero, sin hijos ni otras rentas; escala autonómica igual a la estatal.

## 1. Preparación

In [1]:
import os, subprocess, sys
import pandas as pd
import plotly.graph_objects as go

REPO = "https://github.com/sortaria24-max/08MBID_TrabajoPractico.git"
if not os.path.exists("src/modelo_fiscal.py"):
    if os.path.exists("../src/modelo_fiscal.py"):   # ejecutado desde la carpeta notebooks/
        os.chdir("..")
    else:                                            # Colab: descargamos el repositorio
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
        os.chdir("08MBID_TrabajoPractico")
sys.path.insert(0, "src")
import modelo_fiscal as mf

AZUL, NARANJA, TINTA, TINTA2, GRIS = "#2A78D6", "#EB6834", "#14130F", "#55534C", "#D9D6CC"
COLOR = {"neto": "#9DC1EC", "ss": "#C9C5B8", "irpf": NARANJA}
NOMBRE = {"neto": "Llega a la cuenta", "ss": "Seguridad Social", "irpf": "IRPF"}

def euros(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

def pct(x):
    return f"{x:.1f}".replace(".", ",") + " %"

## 2. Datos

Leemos la curva bruto → neto y añadimos, para cada sueldo, el **tramo más alto** del IRPF que alcanza su base liquidable (se calcula con el mismo modelo fiscal). Comprobamos que Seguridad Social + IRPF + neto suman el bruto.

In [2]:
curva = pd.read_csv("data/processed/curva_bruto_neto.csv")
assert (curva.ss + curva.irpf + curva.neto - curva.bruto).abs().max() < 0.01

def tramo_maximo(bruto, y):
    base, _ = mf.rendimiento_neto(bruto, y)
    if base <= mf.MINIMO_CONTRIBUYENTE:
        return "no paga (mínimo personal)"
    umbrales = mf.umbrales_escala(y)
    i = max(k for k, u in enumerate(umbrales) if base > u)
    return f"{2 * mf.TIPOS_ESTATAL[i]:.0f} %"

curva["tramo"] = [tramo_maximo(b, y) for b, y in zip(curva.bruto, curva.anio)]
for c in ("ss", "irpf", "neto"):
    curva[c + "_pct"] = 100 * curva[c] / curva.bruto

s = pd.read_csv("data/processed/salarios_ccaa.csv")
nac = s[s.ccaa == "Nacional"].set_index("anio")
AÑOS = sorted(curva.anio.unique())
curva[curva.bruto.isin([20000, 30000, 50000])].round(1)

,anio,bruto,ss,irpf,irpf_indexado,neto,tramo,ss_pct,irpf_pct,neto_pct
20,2015,20000,1270.0,2338.2,2338.2,16391.8,24 %,6.4,11.7,82.0
40,2015,30000,1905.0,4939.5,4939.5,23155.5,30 %,6.4,16.5,77.2
80,2015,50000,2747.8,11474.3,11474.3,35777.9,37 %,5.5,22.9,71.6
201,2019,20000,1270.0,2338.2,2243.5,16391.8,24 %,6.4,11.7,82.0
221,2019,30000,1905.0,4939.5,4786.4,23155.5,30 %,6.4,16.5,77.2
261,2019,50000,3101.4,11259.5,10992.1,35639.1,37 %,6.2,22.5,71.3
382,2024,20000,1294.0,2047.5,1553.8,16658.5,24 %,6.5,10.2,83.3
402,2024,30000,1941.0,4928.7,4101.2,23130.3,30 %,6.5,16.4,77.1
442,2024,50000,3235.0,11210.0,9764.9,35555.0,37 %,6.5,22.4,71.1


## 3. Gráfico interactivo

- **Áreas apiladas al 100 %:** de abajo arriba, lo que llega a la cuenta (azul), la Seguridad Social (gris) y el IRPF (naranja). La altura de cada franja es el porcentaje del sueldo bruto que se lleva.
- **Líneas discontinuas:** el salario mediano y el medio del año elegido.
- **Recuadro de arriba:** el reparto de la nómina del salario mediano de ese año.
- **Botones:** cambian de año.

In [3]:
fig = go.Figure()
capas, notas, lineas = {}, {}, {}

for y in AÑOS:
    d = curva[curva.anio == y]
    ids = []
    for c in ("neto", "ss", "irpf"):
        texto = [f"{NOMBRE[c]}: <b>{euros(v)}</b> ({pct(p)})" + (f" · tramo más alto: {t}" if c == "irpf" else "")
                 for v, p, t in zip(d[c], d[c + "_pct"], d.tramo)]
        fig.add_trace(go.Scatter(x=d.bruto, y=d[c], stackgroup="uno", groupnorm="percent", name=NOMBRE[c],
                                 mode="none", fillcolor=COLOR[c], text=texto, hovertemplate="%{text}<extra></extra>",
                                 visible=(y == AÑOS[-1]), showlegend=False))
        ids.append(len(fig.data) - 1)
    capas[y] = ids

    # Rótulos de cada franja, a la derecha
    fila = d[d.bruto == 90000].iloc[0]
    n, ss_ = fila.neto_pct, fila.ss_pct
    rotulos = [dict(x=90000, y=n / 2, text="<b>Llega a la cuenta</b>", font=dict(color=TINTA, size=13)),
               dict(x=90000, y=n + ss_ / 2, text="Seguridad Social", font=dict(color=TINTA, size=11)),
               dict(x=90000, y=n + ss_ + fila.irpf_pct / 2, text="<b>IRPF</b>", font=dict(color="white", size=13))]

    # Salario mediano y medio de ese año
    marcas, formas = [], []
    for col, etiqueta, ancla in (("mediana", "Mediano", "right"), ("media", "Medio", "left")):
        v = nac.loc[y, col]
        formas.append(dict(type="line", x0=v, x1=v, y0=0, y1=100, line=dict(color=TINTA, width=1.5, dash="dash")))
        marcas.append(dict(x=v, y=4, yanchor="bottom", xanchor=ancla, xshift=-4 if ancla == "right" else 4,
                           text=f"{etiqueta}<br><b>{euros(v)}</b>", align=ancla, showarrow=False,
                           font=dict(size=11, color=TINTA), bgcolor="rgba(255,255,255,0.85)", borderpad=3))

    # Recuadro con la nómina del salario mediano
    b = nac.loc[y, "mediana"]
    irpf, ss = mf.irpf_individual(b, y)
    neto = b - ss - irpf
    caja = dict(x=0, y=1.0, xref="paper", yref="paper", xanchor="left", yanchor="bottom", yshift=14, align="left",
                showarrow=False, bgcolor="white", bordercolor=GRIS, borderwidth=1, borderpad=8,
                font=dict(size=13, color=TINTA),
                text=(f"<b>Salario mediano de {y}: {euros(b)} brutos</b><br>"
                      f"Seguridad Social {euros(ss)} ({pct(100 * ss / b)}) · IRPF {euros(irpf)} ({pct(100 * irpf / b)})<br>"
                      f"Llega a la cuenta: <b>{euros(neto)}</b> al año ({pct(100 * neto / b)})"))
    for r in rotulos:
        r.update(showarrow=False, xanchor="center")
    notas[y] = [caja] + rotulos + marcas
    lineas[y] = formas

botones = []
for y in AÑOS:
    vis = [False] * len(fig.data)
    for i in capas[y]:
        vis[i] = True
    botones.append(dict(label=str(y), method="update",
                        args=[{"visible": vis}, {"annotations": notas[y], "shapes": lineas[y]}]))

INICIAL = AÑOS[-1]
fig.update_layout(
    annotations=notas[INICIAL], shapes=lineas[INICIAL],
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=AÑOS.index(INICIAL),
                      x=0, y=1.30, xanchor="left", yanchor="bottom", pad=dict(r=6, t=0),
                      bgcolor="white", bordercolor=GRIS, font=dict(size=12, color=TINTA))],
    xaxis=dict(title="Sueldo bruto anual", range=[10000, 100000], tickvals=list(range(10000, 100001, 10000)),
               ticktext=[f"{v:,.0f}".replace(",", ".") for v in range(10000, 100001, 10000)], hoverformat=",.0f", ticksuffix=" €",
               showgrid=False, zeroline=False, linecolor=TINTA2, fixedrange=True),
    yaxis=dict(title="Parte del sueldo bruto", range=[0, 100], ticksuffix=" %", gridcolor="rgba(255,255,255,0.5)",
               zeroline=False, fixedrange=True),
    hovermode="x unified", hoverlabel=dict(font_size=13, bgcolor="white"),
    font=dict(family="Inter, system-ui, -apple-system, Segoe UI, sans-serif", color=TINTA2, size=12),
    paper_bgcolor="white", plot_bgcolor="white", height=580, separators=",.",
    margin=dict(l=60, r=20, t=160, b=60),
)
fig.show(config={"displaylogo": False})

## 4. Guardar

- `figura5_web.html`: el bloque que se pega en `index.html`, dentro del `<div class="fig-live">` de la Figura 5.
- Para el PDF: el icono de la cámara, arriba a la derecha del gráfico, descarga `figura5.png` con el año que se esté viendo (2024, por defecto). Súbelo a la carpeta `imagenes` del repositorio.

In [4]:
config = {"displaylogo": False, "responsive": True,
          "toImageButtonOptions": {"format": "png", "filename": "figura5", "scale": 2, "width": 1000, "height": 580}}
fragmento = fig.to_html(full_html=False, include_plotlyjs=False, div_id="grafico-figura5",
                        config=config, default_width="100%", default_height="580px")
open("figura5_web.html", "w", encoding="utf-8").write(fragmento)
print("Guardado: figura5_web.html (para pegar en index.html)")

try:
    from google.colab import files
    files.download("figura5_web.html")
except ImportError:
    pass

Guardado: figura5_web.html (para pegar en index.html)
